# 03 · Random Forest

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos).

El cuaderno tiene cuatro etapas, cada una en su propio experimento de MLflow:

| Etapa | Sección | Experimento MLflow |
|---|---|---|
| Búsqueda preliminar: modelo por defecto de scikit-learn y 10 configuraciones | 0 | `bogoprice-random-forest-prebusqueda` |
| Conjuntos de variables con la base elegida | 1 | `bogoprice-random-forest` |
| Un hiperparámetro a la vez | 2 | `bogoprice-random-forest` |
| Ajuste: combinaciones de los dos hiperparámetros que más aportaron | 3 | `bogoprice-random-forest-ajuste` |

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"

# Qué secciones se ejecutan (cada una registra sus corridas en MLflow)
EJECUTAR_PREBUSQUEDA = False   # Sección 0
EJECUTAR_BASE = False          # Sección 1
EJECUTAR_EXPLORACION = False   # Sección 2
EJECUTAR_AJUSTE = True         # Sección 3 y resumen

datos = preparar_datos()
resultados = []


def rf(**p):
    # Base elegida en la búsqueda preliminar (sección 0): cada prueba cambia solo los indicados
    base = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.3, "max_samples": None}
    return RandomForestRegressor(n_jobs=-1, random_state=0, **(base | p)), base | p

## 0. Búsqueda preliminar: de los valores por defecto a la configuración base

El punto de partida es un Random Forest con los **valores por defecto de scikit-learn**: 100 árboles, sin límite de profundidad, hojas de un solo anuncio (`min_samples_leaf=1`), todas las variables en cada corte (`max_features=1.0`) y cada árbol entrenado con una muestra del tamaño completo. 

In [ ]:
if EJECUTAR_PREBUSQUEDA:
    iniciar_mlflow("bogoprice-random-forest-prebusqueda")

    # Valores por defecto de scikit-learn (explícitos para que queden registrados en MLflow)
    DEFECTO = {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1,
               "max_features": 1.0, "max_samples": None}

    # (nombre, tipo, cambios respecto a DEFECTO)
    configuraciones = [
        ("defecto",                "Defecto scikit-learn", {}),
        ("base_anterior",          "Con sentido", {"n_estimators": 400, "min_samples_leaf": 3, "max_features": 0.5}),
        ("hoja5_mf05",             "Con sentido", {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.5}),
        ("hoja10_mf05",            "Con sentido", {"n_estimators": 300, "min_samples_leaf": 10, "max_features": 0.5}),
        ("hoja5_mf03",             "Con sentido", {"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.3}),
        ("hoja10_mf05_muestras05", "Con sentido", {"n_estimators": 300, "min_samples_leaf": 10, "max_features": 0.5,
                                                   "max_samples": 0.5}),
        ("hoja1_mf03",             "Intermedia",  {"n_estimators": 300, "max_features": 0.3}),
        ("profundidad15",          "Intermedia",  {"n_estimators": 300, "max_depth": 15}),
        ("hoja50_mf05",            "Intermedia",  {"n_estimators": 300, "min_samples_leaf": 50, "max_features": 0.5}),
        ("profundidad3",           "Sin sentido", {"n_estimators": 300, "max_depth": 3}),
        ("arboles5",               "Sin sentido", {"n_estimators": 5}),
    ]

    previa = []
    for version in ["v1", "v2", "v2_geo"]:
        for nombre_cfg, tipo, cambios in configuraciones:
            params = DEFECTO | cambios
            modelo = RandomForestRegressor(n_jobs=-1, random_state=0, **params)
            nombre = f"rf_pre_{nombre_cfg}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "RandomForest", "parametro_variado": "prebusqueda", "tipo": tipo, **params},
                        AUTOR)[0]
            previa.append({"configuracion": nombre_cfg, "tipo": tipo, "conjunto": version, **m})
            print(f"{version:7s} {nombre_cfg:24s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %  "
                  f"entrenamiento = {m['mdape_train']:.2f} %")

    previa = pd.DataFrame(previa)
    previa["brecha"] = previa["mdape_nuevos"] - previa["mdape_train"]

In [ ]:
if EJECUTAR_PREBUSQUEDA:
    # Tabla: MdAPE en anuncios nuevos y brecha frente a entrenamiento, por conjunto
    orden = [c[0] for c in configuraciones]
    tabla_previa = (previa.pivot_table(index=["configuracion", "tipo"], columns="conjunto",
                                       values=["mdape_nuevos", "brecha"])
                    .reindex(orden, level=0).round(2))
    display(tabla_previa)

    # Figura: error de entrenamiento vs. anuncios nuevos para cada configuración
    fig, ejes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for eje, version in zip(ejes, ["v1", "v2", "v2_geo"]):
        d = previa[previa["conjunto"] == version].set_index("configuracion").loc[orden]
        y = range(len(d))
        eje.barh([i + 0.2 for i in y], d["mdape_train"], height=0.4, label="Entrenamiento (junio)")
        eje.barh([i - 0.2 for i in y], d["mdape_nuevos"], height=0.4, label="Anuncios nuevos (sept.)")
        eje.axvline(20, color="red", ls="--", lw=1, label="Meta 20 %")
        eje.set_yticks(list(y), [f"{c} ({t})" for c, t in zip(d.index, d["tipo"])])
        eje.invert_yaxis()
        eje.set_xlabel("MdAPE (%)")
        eje.set_title(f"Búsqueda preliminar — {version}")
    ejes[0].legend(loc="lower right")
    fig.tight_layout()

    # Se guarda la figura y la tabla en MLflow, en una corrida de resumen
    mejores_pre = previa.loc[previa.groupby("conjunto")["mdape_nuevos"].idxmin()]
    with tempfile.TemporaryDirectory() as tmp:
        ruta_fig = os.path.join(tmp, "prebusqueda_rf.png")
        ruta_csv = os.path.join(tmp, "prebusqueda_rf.csv")
        fig.savefig(ruta_fig, dpi=130)
        previa.to_csv(ruta_csv, index=False)
        registrar_corrida(
            "rf_pre_resumen",
            {"configuraciones": str(orden)},
            {f"mejor_mdape_nuevos_{fila.conjunto}": float(fila.mdape_nuevos) for fila in mejores_pre.itertuples()},
            artefactos=[ruta_fig, ruta_csv],
            tags={"autor": AUTOR, "modelo": "Resumen"},
        )
    plt.show()

### Elección de la configuración base

A partir de la tabla y la figura se elige la base para las secciones siguientes: **300 árboles, `min_samples_leaf=5`, `max_features=0.3`** (`hoja5_mf03`), la mejor con `v2` (20,15 %) y segunda con `v2_geo` (18,24 %, a 0,32 puntos de la mejor): es la de mejor desempeño combinado en los dos conjuntos. 

## 1. Conjuntos de variables

La configuración base elegida en la sección 0 (300 árboles, `min_samples_leaf=5`, `max_features=0.3`) con los tres conjuntos de variables.

In [ ]:
if EJECUTAR_BASE:
    iniciar_mlflow("bogoprice-random-forest")
    for version in ["v2", "v2_geo"]:
        modelo, params = rf()
        m = evaluar(datos, f"rf_{version}", modelo, version, {"modelo": "RandomForest", **params}, AUTOR)[0]
        resultados.append({"corrida": f"rf_{version}", "conjunto": version,
                           "parametro": "base", "valor": "-", **m})
        print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se partió de la configuración base (300 árboles, `min_samples_leaf=5`, `max_features=0.3`) y se cambió un hiperparámetro a la vez. MdAPE en anuncios nuevos (base: 20,15 % con `v2` y 18,24 % con `v2_geo`):

| Hiperparámetro | Valores probados | `v2` | `v2_geo` | ¿Aporta? |
|---|---|---|---|---|
| `min_samples_leaf` | 1 / 10 / 20 | 21,44 / 20,48 / 20,93 | 18,70 / 18,41 / 19,18 | Sí |
| `max_features` | 0,5 / 1,0 | 20,46 / 21,33 | 18,35 / 18,09 | Sí |
| `max_samples` | 0,5 / 0,7 | 20,13 / 20,02 | 18,39 / 18,28 | Poco |
| `n_estimators` | 100 / 500 | 20,41 / 19,74 | 18,17 / 18,42 | No |
| `max_depth` | 10 / 20 | 20,72 / 20,05 | 19,39 / 18,51 | No|
| `min_samples_split` | 30 | 20,04 | 18,72 | No |
| `bootstrap` | False | 20,39 | 18,74 | No |
| `criterion` | poisson | 20,18 | 18,23 | No |
| `ccp_alpha` | 0,001 | 22,93 | 21,83 | No |

Los parámetros más influyentes son **`min_samples_leaf`** y **`max_features`** , cuyo efecto además cambia según haya o no coordenadas. Esos dos se cruzan en la sección 3.

In [ ]:
if EJECUTAR_EXPLORACION:
    iniciar_mlflow("bogoprice-random-forest")
    pruebas = {
        "n_estimators": [100, 500],
        "max_depth": [10, 20],
        "max_features": [0.5, 1.0],
        "min_samples_leaf": [1, 10, 20],
        "max_samples": [0.5, 0.7],
        "min_samples_split": [30],
        "bootstrap": [False],
        "criterion": ["poisson"],
        "ccp_alpha": [0.001],
    }
    for version in ["v2", "v2_geo"]:
        for parametro, valores in pruebas.items():
            for valor in valores:
                cambios = {parametro: valor}
                # Sin bootstrap no se puede submuestrear: max_samples vuelve a None
                if parametro == "bootstrap" and valor is False:
                    cambios["max_samples"] = None
                modelo, params = rf(**cambios)
                nombre = f"rf_{parametro}{valor}_{version}"
                m = evaluar(datos, nombre, modelo, version,
                            {"modelo": "RandomForest", "parametro_variado": parametro, **params}, AUTOR)[0]
                resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                                   "valor": str(valor), **m})
    tabla = pd.DataFrame(resultados)
    display(tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                              values="mdape_nuevos").round(2))

## 3. Ajuste: `min_samples_leaf` × `max_features`

Se prueban todas las combinaciones de los dos hiperparámetros más influyentes en la sección 2, con y sin coordenadas: `min_samples_leaf` (3, 5, 10, 20) × `max_features` (0,2, 0,3, 0,5, 0,7, 1,0). Los demás quedan en la base (300 árboles, `max_samples=None`).

In [ ]:
if EJECUTAR_AJUSTE:
    iniciar_mlflow("bogoprice-random-forest-ajuste")

    hojas = [3, 5, 10, 20]
    variables_corte = [0.2, 0.3, 0.5, 0.7, 1.0]

    grilla = []
    for version in ["v2", "v2_geo"]:
        for hoja in hojas:
            for mf in variables_corte:
                modelo, params = rf(min_samples_leaf=hoja, max_features=mf)
                nombre = f"rf_leaf{hoja}_mf{mf}_{version}"
                m = evaluar(datos, nombre, modelo, version,
                            {"modelo": "RandomForest", "parametro_variado": "grilla", **params}, AUTOR)[0]
                grilla.append({"conjunto": version, "min_samples_leaf": hoja, "max_features": mf, **m})
                print(f"{version:7s} hoja={hoja:<3d} max_features={mf:<4} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

    grilla = pd.DataFrame(grilla)

In [ ]:
if EJECUTAR_AJUSTE:
    # Mapa de calor del MdAPE en anuncios nuevos para cada conjunto de variables
    fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
    for eje, version in zip(ejes, ["v2", "v2_geo"]):
        tabla = grilla[grilla["conjunto"] == version].pivot(
            index="min_samples_leaf", columns="max_features", values="mdape_nuevos")
        eje.imshow(tabla.values, cmap="RdYlGn_r", aspect="auto")
        eje.set_xticks(range(len(tabla.columns)), [f"{c:g}" for c in tabla.columns])
        eje.set_yticks(range(len(tabla.index)), tabla.index)
        for i in range(tabla.shape[0]):
            for j in range(tabla.shape[1]):
                eje.text(j, i, f"{tabla.values[i, j]:.2f}", ha="center", va="center", color="black")
        eje.set_xlabel("max_features (fracción de variables por corte)")
        eje.set_ylabel("min_samples_leaf")
        eje.set_title(f"MdAPE anuncios nuevos (%) — {version}")
    fig.tight_layout()

    # Se guarda la figura y la tabla en MLflow, en una corrida de resumen
    mejores = grilla.loc[grilla.groupby("conjunto")["mdape_nuevos"].idxmin()]
    with tempfile.TemporaryDirectory() as tmp:
        ruta_fig = os.path.join(tmp, "grilla_rf_mdape_nuevos.png")
        ruta_csv = os.path.join(tmp, "grilla_rf.csv")
        fig.savefig(ruta_fig, dpi=130)
        grilla.to_csv(ruta_csv, index=False)
        registrar_corrida(
            "rf_resumen_grilla",
            {"min_samples_leaf": str(hojas), "max_features": str(variables_corte)},
            {f"mejor_mdape_nuevos_{fila.conjunto}": float(fila.mdape_nuevos) for fila in mejores.itertuples()},
            artefactos=[ruta_fig, ruta_csv],
            tags={"autor": AUTOR, "modelo": "Resumen"},
        )
    plt.show()

## Resumen

Las tres mejores combinaciones de cada conjunto de variables.

In [ ]:
if EJECUTAR_AJUSTE:
    columnas = ["conjunto", "min_samples_leaf", "max_features", "mdape_nuevos", "mdape", "mdape_train"]
    display(grilla.sort_values("mdape_nuevos").groupby("conjunto").head(3)[columnas].round(2))